# Helmet detection - train gộp dữ liệu (Kaggle, chạy nền bằng Save & Run All)

**Chuẩn bị trước khi bấm Save Version**
1. Settings: **Accelerator = GPU T4**, **Internet = On**.
2. Add-ons → Secrets: gắn secret `ROBOFLOW_API_KEY` (bật *Attached*).
3. Add Data: dataset chứa **file zip project** (`helmet-tracking.zip`) và dataset chứa **video** `.mp4`.
4. Bấm **Save Version → Save & Run All (Commit)**. Notebook chạy nền khoảng 1,5-2 giờ, không cần mở tab. Kết quả nằm ở tab **Output** của version đã lưu (`best_merged.pt`, thư mục `figures`, `runs`).

## 1. Lấy project

In [ ]:
import os, glob, shutil, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
PROJ = WORK / "helmet-tracking"

# Kaggle thường tự giải nén file zip khi tạo dataset, nên thử tìm thư mục có sẵn trước
cands = glob.glob("/kaggle/input/**/src/train.py", recursive=True)
if cands:
    src_root = os.path.dirname(os.path.dirname(cands[0]))
    shutil.rmtree(PROJ, ignore_errors=True)
    shutil.copytree(src_root, PROJ)
else:
    zips = glob.glob("/kaggle/input/**/helmet-tracking*.zip", recursive=True) or glob.glob("/kaggle/input/**/*.zip", recursive=True)
    assert zips, "Không thấy project trong /kaggle/input - kiểm tra đã Add Data chưa"
    tmp = WORK / "proj_tmp"
    shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(zips[0]) as z:
        for info in z.infolist():
            name = info.filename.replace("\\", "/")
            if ".." in name.split("/"):
                continue
            target = tmp / name
            if name.endswith("/"):
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(info) as s, open(target, "wb") as o:
                    shutil.copyfileobj(s, o)
    c2 = glob.glob(str(tmp / "**/src/train.py"), recursive=True)
    assert c2, "Zip không chứa src/train.py"
    shutil.rmtree(PROJ, ignore_errors=True)
    shutil.copytree(os.path.dirname(os.path.dirname(c2[0])), PROJ)
    shutil.rmtree(tmp, ignore_errors=True)

os.chdir(PROJ)
print("Thư mục làm việc:", os.getcwd())
print(sorted(os.listdir(".")))

## 2. Cài thư viện và kiểm tra môi trường

In [ ]:
!pip -q install "ultralytics>=8.3.0" roboflow lap

In [ ]:
import os, glob
import urllib.request, urllib.error
import torch

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ - bật Accelerator trong Settings")
PROJ = "/kaggle/working/helmet-tracking"
print("Project:", "OK" if os.path.isdir(f"{PROJ}/src") else "THIẾU")

VIDEO = (sorted(glob.glob("/kaggle/input/**/*.mp4", recursive=True)) or [None])[0]
print("Video:", VIDEO)
assert VIDEO, "Không thấy video - Add Data dataset chứa file .mp4"

from kaggle_secrets import UserSecretsClient
print("Secret ROBOFLOW_API_KEY:", "OK" if UserSecretsClient().get_secret("ROBOFLOW_API_KEY") else "RỖNG")

try:
    urllib.request.urlopen("https://github.com", timeout=10)
    print("Internet: OK")
except urllib.error.HTTPError:
    print("Internet: OK")

## 3. Tải bộ Mapua

In [ ]:
import os, glob, shutil, collections
import yaml

WORKSPACE = "mapua-university-n7qm7"
PROJECT = "motorcycle-helmet-7d2gr"
VERSION = 1                     # nếu báo lỗi version, đổi thành 2
DEST = "/kaggle/working/mapua_raw"

from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

shutil.rmtree(DEST, ignore_errors=True)
API_KEY = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
Roboflow(api_key=API_KEY).workspace(WORKSPACE).project(PROJECT).version(VERSION) \
    .download("yolov8", location=DEST)

names = yaml.safe_load(open(f"{DEST}/data.yaml"))["names"]
if isinstance(names, list):
    names = dict(enumerate(names))
print("\nLớp:", names)
print("Số ảnh và số box theo lớp (lần trước: train 2270, valid 145, test 139 ảnh):")
for s in ("train", "valid", "test"):
    if not os.path.isdir(f"{DEST}/{s}/images"):
        continue
    cnt = collections.Counter()
    for lf in glob.glob(f"{DEST}/{s}/labels/*.txt"):
        for line in open(lf):
            if line.strip():
                cnt[int(line.split()[0])] += 1
    n = len(glob.glob(f"{DEST}/{s}/images/*.*"))
    print(f"  {s:<6} {n:>5} ảnh | " + " | ".join(f"{names[c]}: {cnt.get(c, 0)}" for c in sorted(names)))

## 4. Gộp dữ liệu

In [ ]:
import os, glob, shutil, collections
import cv2, yaml
from pathlib import Path

PROJ = "/kaggle/working/helmet-tracking"
MAPUA = "/kaggle/working/mapua_raw"          # đã tải ở ô A
OLD = f"{PROJ}/data/dataset"                 # bộ cũ (Dendel Pino)
OUT = "/kaggle/working/merged"
KEEP_INVALID = True      # True: giữ Invalid làm lớp thứ 3 (bỏ qua khi đếm); False: xóa các box Invalid
OVERSAMPLE = 3           # ảnh có lớp Without Helmet xuất hiện bấy nhiêu lần trong tập train
W_OUT, H_OUT = 1280, 720 # trả ảnh Mapua về tỉ lệ 16:9

os.chdir(PROJ)
assert os.path.isdir(f"{MAPUA}/train/images"), "Chưa có bộ Mapua - chạy lại ô A trước"


def ensure_old():
    """Tải lại bộ cũ nếu phiên đã bị đặt lại."""
    if os.path.isdir(f"{OLD}/images/train"):
        return
    print("Tải lại bộ dữ liệu cũ từ Roboflow...")
    from kaggle_secrets import UserSecretsClient
    from roboflow import Roboflow
    key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
    Roboflow(api_key=key).workspace("dendel-pino-27gsx").project("motorcycle-helmet-fhsbn") \
        .version(1).download("yolov8", location="/kaggle/working/rf_old")
    RF = Path("/kaggle/working/rf_old")
    for src, dst in (("train", "train"), ("valid", "val"), ("test", "test")):
        if not (RF / src / "images").exists():
            continue
        os.makedirs(f"{OLD}/images/{dst}", exist_ok=True)
        os.makedirs(f"{OLD}/labels/{dst}", exist_ok=True)
        for f in (RF / src / "images").iterdir():
            shutil.copy(f, f"{OLD}/images/{dst}/{f.name}")
        for f in (RF / src / "labels").iterdir():
            shutil.copy(f, f"{OLD}/labels/{dst}/{f.name}")


ensure_old()

cfg = yaml.safe_load(open(f"{MAPUA}/data.yaml"))
mnames = cfg["names"]
if isinstance(mnames, list):
    mnames = dict(enumerate(mnames))


def to_new(name):
    n = name.lower()
    if "invalid" in n:
        return 2 if KEEP_INVALID else None
    if n.startswith("no") or "without" in n:
        return 1
    if "full" in n or "half" in n:
        return 0
    return None


mapping = {c: to_new(n) for c, n in mnames.items()}
print("Đổi nhãn Mapua:", {mnames[c]: mapping[c] for c in mnames})

shutil.rmtree(OUT, ignore_errors=True)
for d in ("train", "val", "test_old", "test_mapua"):
    os.makedirs(f"{OUT}/images/{d}")
    os.makedirs(f"{OUT}/labels/{d}")
counts = collections.defaultdict(collections.Counter)
imgcount = collections.Counter()


def write_label(dst, name, lines):
    open(f"{OUT}/labels/{dst}/{name}.txt", "w").write("\n".join(lines))
    for l in lines:
        counts[dst][int(l.split()[0])] += 1
    imgcount[dst] += 1


for src, dst in (("train", "train"), ("valid", "val"), ("test", "test_mapua")):
    for ip in glob.glob(f"{MAPUA}/{src}/images/*.*"):
        stem = os.path.splitext(os.path.basename(ip))[0]
        img = cv2.imread(ip)
        if img is None:
            continue
        img = cv2.resize(img, (W_OUT, H_OUT), interpolation=cv2.INTER_CUBIC)
        name = f"mapua_{stem}"
        cv2.imwrite(f"{OUT}/images/{dst}/{name}.jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 95])
        lines = []
        lf = f"{MAPUA}/{src}/labels/{stem}.txt"
        if os.path.exists(lf):
            for line in open(lf):
                p = line.split()
                if len(p) >= 5 and mapping.get(int(p[0])) is not None:
                    lines.append(" ".join([str(mapping[int(p[0])])] + p[1:5]))
        write_label(dst, name, lines)

for src, dst in (("train", "train"), ("val", "val"), ("test", "test_old")):
    for ip in glob.glob(f"{OLD}/images/{src}/*.*"):
        stem, ext = os.path.splitext(os.path.basename(ip))
        name = f"old_{stem}"
        shutil.copy(ip, f"{OUT}/images/{dst}/{name}{ext}")
        lines = []
        lf = f"{OLD}/labels/{src}/{stem}.txt"
        if os.path.exists(lf):
            for line in open(lf):
                p = line.split()
                if len(p) >= 5:
                    lines.append(" ".join(p[:5]))
        write_label(dst, name, lines)

# nhân bản ảnh có lớp Without Helmet trong tập train
lst = []
for ip in sorted(glob.glob(f"{OUT}/images/train/*.*")):
    lf = ip.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    has_no = any(l.split()[0] == "1" for l in open(lf) if l.strip())
    lst += [ip] * (OVERSAMPLE if has_no else 1)
open(f"{OUT}/train_os.txt", "w").write("\n".join(lst))

names3 = {0: "With Helmet", 1: "Without Helmet"}
if KEEP_INVALID:
    names3[2] = "Invalid"
base = {"path": OUT, "names": names3}
yaml.safe_dump({**base, "train": "train_os.txt", "val": "images/val", "test": "images/test_mapua"},
               open(f"{PROJ}/data/merged.yaml", "w"))
for tag in ("old", "mapua"):
    yaml.safe_dump({**base, "train": f"images/test_{tag}", "val": f"images/test_{tag}",
                    "test": f"images/test_{tag}"}, open(f"{PROJ}/data/eval_{tag}.yaml", "w"))

print("\nSố ảnh và số box từng tập:")
for d in ("train", "val", "test_old", "test_mapua"):
    print(f"  {d:<11} {imgcount[d]:>5} ảnh | " + " | ".join(
        f"{names3[c]}: {counts[d].get(c, 0)}" for c in sorted(names3)))
print(f"Danh sách train sau khi nhân bản ảnh Without Helmet: {len(lst)} mẫu")

## 5. Train (khoảng 1 giờ)

In [ ]:
import os
from pathlib import Path

os.chdir("/kaggle/working/helmet-tracking")

# vá đường dẫn lưu kết quả (chạy nhiều lần không sao)
p = Path("src/train.py")
s = p.read_text()
if "from pathlib import Path" not in s:
    s = s.replace("import argparse\n", "import argparse\nfrom pathlib import Path\n", 1)
s = s.replace('project="runs/train"', 'project=str(Path("runs/train").resolve())')
p.write_text(s)
p = Path("src/evaluate.py")
p.write_text(p.read_text().replace('project="runs/eval"', 'project=str(Path("runs/eval").resolve())'))

!python -m src.train --model yolov8n.pt --data data/merged.yaml --imgsz 1280 --batch 8 --epochs 40 --patience 10 --name merged_n_1280 --device 0

## 6. Đánh giá trên val gộp, test bộ cũ và test Mapua; lưu model và hình

In [ ]:
import os, glob, shutil

os.chdir("/kaggle/working/helmet-tracking")
cands = glob.glob("/kaggle/working/**/merged_n_1280*/weights/best.pt", recursive=True)
assert cands, "Không thấy model - bước train có thể đã lỗi"
W2 = max(cands, key=os.path.getmtime)
print("Model mới:", W2)
shutil.copy(W2, "/kaggle/working/best_merged.pt")

for title, data, split in (("VAL gộp", "data/merged.yaml", "val"),
                           ("TEST bộ cũ", "data/eval_old.yaml", "test"),
                           ("TEST Mapua", "data/eval_mapua.yaml", "test")):
    print(f"\n######## {title} ########")
    get_ipython().system(f'python -m src.evaluate --weights "{W2}" --data {data} --split {split} --imgsz 1280')

# gom các hình dùng cho README
R = os.path.dirname(os.path.dirname(W2))
os.makedirs("/kaggle/working/figures", exist_ok=True)
for f in ("results.png", "confusion_matrix_normalized.png", "BoxPR_curve.png", "BoxF1_curve.png",
          "val_batch0_pred.jpg", "val_batch0_labels.jpg", "val_batch1_pred.jpg", "val_batch1_labels.jpg"):
    if os.path.exists(os.path.join(R, f)):
        shutil.copy(os.path.join(R, f), "/kaggle/working/figures/")
print("\nCác hình đã lưu:", sorted(os.listdir("/kaggle/working/figures")))

## 7. Tracking trên video: thử vạch đếm và ngưỡng confidence
Mỗi dòng in ra gồm số người đếm được (tổng và 5 giây đầu) và số track đã tạo ra (số này lớn bất thường nghĩa là track bị đứt nhiều).

In [ ]:
import os, sys, glob, csv, importlib
from collections import Counter
from pathlib import Path

PROJ = "/kaggle/working/helmet-tracking"
os.chdir(PROJ)
sys.path.insert(0, PROJ)

# vá track_count.py: thêm tham số classes (bỏ lớp Invalid) và trả thêm số track đã tạo ra
p = Path("src/track_count.py")
s = p.read_text()
if "classes=None" not in s:
    s = s.replace("device=None, progress=None):", "device=None, progress=None, classes=None):")
    s = s.replace("device=device, verbose=False)", "device=device, classes=classes, verbose=False)")
if '"tracks"' not in s:
    s = s.replace('return {"output": str(out_path), "counts": dict(totals), "frames": n}',
                  'return {"output": str(out_path), "counts": dict(totals), "frames": n, "tracks": len(votes)}')
p.write_text(s)
assert "classes=classes" in s and '"tracks"' in s, "Vá track_count.py không thành công - kiểm tra nội dung file src/track_count.py"
import src.track_count as tc
importlib.reload(tc)

W2 = "/kaggle/working/best_merged.pt"
assert os.path.exists(W2), "Không thấy best_merged.pt - chạy lại ô 3 hoặc gán W2 = đường dẫn model mới"
VIDEO = globals().get("VIDEO") or (glob.glob("/kaggle/input/**/*.mp4", recursive=True) or [None])[0]
assert VIDEO, "Không thấy video"

# (conf, vị trí vạch): conf thấp cho ByteTrack dùng được cả các phát hiện yếu để nối track
for conf, line_y in ((0.25, 0.65), (0.1, 0.55), (0.1, 0.65), (0.1, 0.75)):
    tag = f"c{int(conf * 100)}_y{int(line_y * 100)}"
    res = tc.process_video(W2, VIDEO, output=f"runs/track/{tag}.mp4", conf=conf, imgsz=1280,
                           line_y=line_y, events_csv=f"runs/track/{tag}.csv", classes=[0, 1])
    rows = list(csv.DictReader(open(f"runs/track/{tag}.csv")))
    early = dict(Counter(r["class"] for r in rows if float(r["time_s"]) <= 5))
    print(f"conf={conf} vạch={line_y}: tổng={res['counts']} | 5 giây đầu={early} | số track tạo ra={res['tracks']}")

## 8. Xem vài khung hình của một cấu hình

In [ ]:
import os, cv2
from IPython.display import Image, display

os.chdir("/kaggle/working/helmet-tracking")
path_v = "runs/track/c10_y65.mp4"
cap = cv2.VideoCapture(path_v)
m = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
for k, f in enumerate((0.25, 0.5, 0.75)):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(m * f))
    ok, fr = cap.read()
    if ok:
        p = f"/kaggle/working/figures/track_frame_{k}.jpg"
        cv2.imwrite(p, cv2.resize(fr, (960, 540)))
        display(Image(p))
cap.release()

## 9. Dọn dẹp để thư mục Output gọn (chỉ giữ model, hình và kết quả)

In [ ]:
import shutil, os
for d in ("/kaggle/working/merged", "/kaggle/working/mapua_raw", "/kaggle/working/rf_old",
          "/kaggle/working/rf_dataset", "/kaggle/working/helmet-tracking/data/dataset"):
    shutil.rmtree(d, ignore_errors=True)
get_ipython().system("ls -la /kaggle/working && du -sh /kaggle/working")